## K-Nearest Neighbors

In [24]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [25]:
df = pd.read_csv("data/algerian_forest_fires_proj4.csv")
df.head()

,region,temp_c,rel_humidity_percent,wind_speed_kmh,rainfall_mm,ffmc,dmc,dc,isi,bui,fwi,classes
0,Bejaia,29.0,57.0,18.0,0.0,65.7,3.4,7.6,1.3,3.4,0.5,not fire
1,Bejaia,29.0,61.0,13.0,1.3,64.4,4.1,7.6,1.0,3.9,0.4,not fire
2,Bejaia,26.0,82.0,22.0,13.1,47.1,2.5,7.1,0.3,2.7,0.1,not fire
3,Bejaia,25.0,89.0,13.0,2.5,28.6,1.3,6.9,0.0,1.7,0.0,not fire
4,Bejaia,27.0,77.0,16.0,0.0,64.8,3.0,14.2,1.2,3.9,0.5,not fire


In [26]:
df.shape

(243, 12)

### Creating the KNN

In [27]:
tested_df = df.drop('region', axis=1).copy()
tested_df.head()

,temp_c,rel_humidity_percent,wind_speed_kmh,rainfall_mm,ffmc,dmc,dc,isi,bui,fwi,classes
0,29.0,57.0,18.0,0.0,65.7,3.4,7.6,1.3,3.4,0.5,not fire
1,29.0,61.0,13.0,1.3,64.4,4.1,7.6,1.0,3.9,0.4,not fire
2,26.0,82.0,22.0,13.1,47.1,2.5,7.1,0.3,2.7,0.1,not fire
3,25.0,89.0,13.0,2.5,28.6,1.3,6.9,0.0,1.7,0.0,not fire
4,27.0,77.0,16.0,0.0,64.8,3.0,14.2,1.2,3.9,0.5,not fire


In [28]:
fire_data_x = tested_df.drop('classes', axis=1)
fire_data_y = tested_df['classes']

cross_validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [29]:
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

parameters = {
    'knn__n_neighbors': range(1, 26),
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['manhattan', 'euclidean']
}

In [30]:
knn_grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=parameters,
    scoring='accuracy',
    cv=cross_validation,
    n_jobs=-1,
    verbose=2
)
knn_grid.fit(fire_data_x, fire_data_y)
print('Best parameters ', knn_grid.best_params_)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END knn__metric=manhattan, knn__n_neighbors=1, knn__weights=distance; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=1, knn__weights=distance; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=1, knn__weights=distance; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=2, knn__weights=uniform; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=2, knn__weights=uniform; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=1, knn__weights=uniform; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=2, knn__weights=uniform; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=2, knn__weights=uniform; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=2, knn__weights=uniform; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=2, knn__weights=distance; total time=   0.0s
[CV

In [31]:
BEST_KNN_CLF = knn_grid.best_estimator_

In [32]:
final_scores = cross_val_score(
    BEST_KNN_CLF,
    fire_data_x,
    fire_data_y,
    scoring='accuracy',
    cv=cross_validation,
)
print(final_scores)
print(final_scores.mean())
print(final_scores.std())

[0.89795918 0.95918367 0.95918367 0.9375     0.95833333]
0.9424319727891156
0.02373286980471338


In [33]:
knn_predict_y = cross_val_predict(BEST_KNN_CLF, fire_data_x, fire_data_y, cv=cross_validation)

In [34]:
classification_accuracy = round(accuracy_score(fire_data_y, knn_predict_y) * 100, 4)
confusion_matrix = confusion_matrix(fire_data_y, knn_predict_y)
per_class_classification = classification_report(fire_data_y, knn_predict_y)

In [35]:
print(classification_accuracy)
print(confusion_matrix)
print(per_class_classification)

94.2387
[[133   4]
 [ 10  96]]
              precision    recall  f1-score   support

        fire       0.93      0.97      0.95       137
    not fire       0.96      0.91      0.93       106

    accuracy                           0.94       243
   macro avg       0.95      0.94      0.94       243
weighted avg       0.94      0.94      0.94       243

